# 🌍 Động đất Toàn cầu — Phân tích Xu hướng & Dự đoán Số trận (1990–2013)

**Sinh viên:** ................ &nbsp;|&nbsp; **MSSV:** ................ &nbsp;|&nbsp; **Môn học:** Phân tích Dữ liệu và Trí tuệ Nhân tạo

**Nguồn dữ liệu:** USGS — United States Geological Survey (Earthquake Catalog API), các trận có độ lớn M ≥ 5.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

ACCENT = "#FF8C00"
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#0d1117",
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#e6edf3",
    "text.color": "#e6edf3",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})
PALETTE = sns.dark_palette(ACCENT, n_colors=12, reverse=True)
sns.set_palette(PALETTE)

## 1. Thu thập & Khảo sát Dữ liệu

In [ ]:
DATA_URL = ("https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
            "&starttime=1990-01-01&endtime=2013-12-31&minmagnitude=5.5&orderby=time-asc")

df = pd.read_csv(DATA_URL)
df.shape

In [ ]:
df.dtypes

In [ ]:
df[["time", "latitude", "longitude", "depth", "mag", "magType", "place", "type"]].head()

In [ ]:
df[["time", "latitude", "longitude", "depth", "mag", "magType", "place", "type"]].isnull().sum()

## 2. Tiền xử lý & Làm sạch Dữ liệu

In [ ]:
df = df[["time", "latitude", "longitude", "depth", "mag", "magType", "place", "type"]].copy()

df["datetime"] = pd.to_datetime(df["time"], utc=True, errors="coerce").dt.tz_localize(None)
df["mag"] = pd.to_numeric(df["mag"], errors="coerce")
df["depth"] = pd.to_numeric(df["depth"], errors="coerce")

# Chỉ giữ động đất tự nhiên (loại bỏ nổ mìn, thử hạt nhân...)
df = df[df["type"] == "earthquake"]
df = df.dropna(subset=["datetime", "mag", "depth", "latitude", "longitude"])
df = df.drop_duplicates()

df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["month_name"] = df["datetime"].dt.month_name()

# Khu vực = phần sau dấu phẩy cuối cùng của cột place (hoặc cả chuỗi nếu không có dấu phẩy)
df["region"] = df["place"].fillna("Unknown").str.split(",").str[-1].str.strip()

# Phân nhóm độ sâu
df["depth_class"] = pd.cut(df["depth"], bins=[-np.inf, 70, 300, np.inf],
                           labels=["Nông (<70 km)", "Trung bình (70-300 km)", "Sâu (>300 km)"])

df = df[(df["year"] >= 1990) & (df["year"] <= 2013)]
df.shape

## 3. Thống kê Mô tả

In [ ]:
df[["year", "mag", "depth"]].describe()

In [ ]:
df["depth_class"].value_counts()

In [ ]:
df["magType"].value_counts()

In [ ]:
df["region"].value_counts().head(10)

## 4. Phân tích Khám phá Dữ liệu (EDA)

In [ ]:
yearly = df.groupby("year").size()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.index, yearly.values, color=ACCENT, linewidth=2.2, marker="o", markersize=4)
ax.fill_between(yearly.index, yearly.values, color=ACCENT, alpha=0.08)
ax.set_title("Số trận Động đất M ≥ 5.5 theo Năm (1990–2013)")
ax.set_xlabel("Năm")
ax.set_ylabel("Số trận động đất")
plt.tight_layout()
plt.show()

In [ ]:
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
monthly = df.groupby("month_name").size().reindex(month_order)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=monthly.index, y=monthly.values, hue=monthly.index,
            palette=PALETTE, legend=False, ax=ax)
ax.set_title("Số trận Động đất theo Tháng (gộp 1990–2013)")
ax.set_xlabel("Tháng")
ax.set_ylabel("Số trận động đất")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(df["mag"], bins=25, color=ACCENT, ax=ax)
ax.set_title("Phân bố Độ lớn Động đất (Magnitude)")
ax.set_xlabel("Độ lớn (Mw)")
ax.set_ylabel("Số trận")
plt.tight_layout()
plt.show()

In [ ]:
top_regions = df["region"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=top_regions.values, y=top_regions.index, hue=top_regions.index,
            palette=PALETTE, legend=False, ax=ax)
ax.set_title("Top 10 Khu vực Có Nhiều Động đất Nhất")
ax.set_xlabel("Số trận động đất")
ax.set_ylabel("Khu vực")
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(index="year", columns="month", values="datetime", aggfunc="count")

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap=sns.dark_palette(ACCENT, as_cmap=True), linewidths=0.4,
            linecolor="#0d1117", ax=ax, cbar_kws={"label": "Số trận động đất"})
ax.set_title("Mật độ Động đất theo Năm x Tháng")
ax.set_xlabel("Tháng")
ax.set_ylabel("Năm")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
sc = ax.scatter(df["longitude"], df["latitude"], c=df["mag"], cmap="YlOrRd",
                s=(df["mag"] - 5) ** 3 * 2, alpha=0.6, linewidths=0)
plt.colorbar(sc, ax=ax, label="Độ lớn (Mw)")
ax.set_title("Phân bố Địa lý Động đất (Vành đai lửa Thái Bình Dương)")
ax.set_xlabel("Kinh độ")
ax.set_ylabel("Vĩ độ")
plt.tight_layout()
plt.show()

## 5. Mô hình Dự đoán Xu hướng

In [ ]:
ts = df.set_index("datetime").resample("MS").size()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts.index, ts.values, color=ACCENT, linewidth=1.6)
ax.set_title("Chuỗi Thời gian: Số trận Động đất theo Tháng")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Số trận / tháng")
plt.tight_layout()
plt.show()

In [ ]:
X = np.arange(len(ts)).reshape(-1, 1)
y = ts.values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2, mse

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
idx_train = ts.index[:len(y_train)]
idx_test = ts.index[len(y_train):]

ax.plot(idx_train, y_train, color="#8b949e", linewidth=1.4, label="Train (thực tế)")
ax.plot(idx_test, y_test, color=ACCENT, linewidth=1.8, label="Test (thực tế)")
ax.plot(idx_test, y_pred, color="#4dc3ff", linewidth=1.8, linestyle="--", label="Dự đoán")
ax.set_title(f"Dự đoán Số trận Động đất theo Tháng  (R² = {r2:.3f})")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Số trận / tháng")
ax.legend(facecolor="#0d1117", edgecolor="#30363d")
plt.tight_layout()
plt.show()

## 6. Tổng kết

In [ ]:
summary = {
    "Tổng số bản ghi sau làm sạch": len(df),
    "Khoảng thời gian": f"{df['year'].min()}-{df['year'].max()}",
    "Độ lớn trung bình": round(df["mag"].mean(), 2),
    "Độ lớn lớn nhất": df["mag"].max(),
    "Nhóm độ sâu phổ biến nhất": df["depth_class"].value_counts().idxmax(),
    "Tháng nhiều động đất nhất": monthly.idxmax(),
    "Khu vực đứng đầu": top_regions.idxmax(),
    "R² mô hình dự đoán": round(r2, 3),
    "MSE mô hình dự đoán": round(mse, 1),
}
pd.Series(summary).to_frame("Giá trị")

### Nhận xét

- Số trận động đất M ≥ 5.5 mỗi năm dao động quanh một mức tương đối ổn định, không có xu hướng tăng/giảm rõ rệt; đây là hiện tượng tự nhiên chứ không phải do hoạt động báo cáo như dữ liệu UFO.
- Động đất tập trung chủ yếu dọc **Vành đai lửa Thái Bình Dương** (Nhật Bản, Indonesia, Chile, Tonga, Papua New Guinea...).
- Mô hình hồi quy tuyến tính chỉ nắm được xu hướng tuyến tính nên R² thấp là điều dễ hiểu; có thể cải thiện bằng mô hình chuỗi thời gian (ARIMA, Poisson regression).